# Chapter2: Tokens and Embeddings

some links:
- https://platform.openai.com/tokenizer

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

d:\2026-courses\LLMs-Handson\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0405 16:14:17.038000 37184 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.
Loading checkpoint shards: 100%|██████████| 2/2 [00:17<00:00,  8.95s/it]
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


We can now proceed to the actual generation. We first declare our prompt, then tokenize it, then pass those tokens to the model, which gnerates its output. In this case, we're asking the model to only generate 20 new tokens

In [2]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap.Explain how it happened.<|assistant|>"

# Tokenize the prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=20
)



You are not running the flash-attention implementation, expect numerical differences.


In [3]:
# Print the output
print(tokenizer.decode(generation_output[0]))

Write an email apologizing to Sarah for the tragic gardening mishap.Explain how it happened.<|assistant|> Subject: Sincere Apologies for the Gardening Mishap


Dear


the text generate by the model: Subject: Sincere Apologies for the Gardening Mishap

Dear

In [4]:
print(input_ids)

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889,  9544,  7420,   920,
           372,  9559, 29889, 32001]], device='cuda:0')


In [5]:
for id in input_ids[0]:
    print(tokenizer.decode(id))

Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.
Exp
lain
how
it
happened
.
<|assistant|>


This is how the tokenizer broke down the input prompt. Notice the following:

- The first token is ID 1 (s), a special token indicating the begenning of the text.
- Some tokens are complete words (e.g., Write, an, email).
- Some tokens are parts of words (apolog, izing, trag, ic)
- Punctuation characters are their own token.

In [6]:
generation_output

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889,  9544,  7420,   920,
           372,  9559, 29889, 32001,  3323,   622, 29901,   317,  3742,   406,
          6225, 11763,   363,   278, 19906,   292,   341,   728,   481,    13,
            13,    13, 29928,   799]], device='cuda:0')

In [7]:
for id in generation_output[0]:
    print(tokenizer.decode(id))

Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.
Exp
lain
how
it
happened
.
<|assistant|>
Sub
ject
:
S
inc
ere
Ap
ologies
for
the
Garden
ing
M
ish
ap






D
ear


In [8]:
print(tokenizer.decode(3323))
print(tokenizer.decode(622))
print(tokenizer.decode([3323, 622]))
print(tokenizer.decode(29901))

Sub
ject
Subject
:


How does the tokenizer break down text?

1- Tokenization methods:

- GPT: BPE (byte pair encoding)
- BERT: WordPiec

2- After choosing the tokenization method, we need to make a number of tokenizer design choice like vocabulary size and what special tokens to use.

3-  the tokenizer needs to be trained on a specific dataset to establish the best vocabulary it can use to represent that dataset. Even if we set the same methods and parameters, a tokenizer trained on an English text dataset will be different from another trained on a code dataset or a multilingual text dataset.

In this context, we have three major factors that dictate the tokens that appear within a tokenizer: the tokenization method, the parameters and special tokens we use to initialize the tokenizer, and the dataset the tokenizer is trained on

## Comparing Trained LLM tokens

This will allow us to see how each tokenizer deals with a number of different kinds of tokens:
- Capitalization.
- Languages other than English.
- Emojis.
- Programming code with keywords and whitespaces often used for indentation (in languages like Python for example).
- Numbers and digits.
- Special tokens. These are unique tokens that have a role other than representing text. They include tokens that indicate the beginning of the text, or the end of the text (which is the way the model signals to the system that it has completed this generation), or other functions as we’ll see.


In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import math

# Bright, vivid colors optimized for dark mode terminals
colors_list = [
    '100;200;255',  # Bright cyan
    '255;150;100',  # Bright coral
    '150;255;150',  # Bright green
    '255;200;100',  # Bright orange
    '200;150;255',  # Bright purple
    '255;100;150',  # Bright pink
    '100;255;200',  # Bright aqua
    '255;255;100',  # Bright yellow
    '150;200;255',  # Bright sky blue
    '255;180;200',  # Bright rose
    '200;255;150',  # Bright lime
    '180;150;255'   # Bright lavender
]

def show_tokens(sentence, tokenizer_name, show_ids=True, show_stats=True):
    """
    Enhanced token visualization with colors, IDs, and statistics.
    Optimized for dark mode terminals with high contrast colors.
    
    Args:
        sentence: Text to tokenize
        tokenizer_name: Name of the tokenizer to use
        show_ids: Whether to show token IDs
        show_stats: Whether to show statistics
    """
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids
    tokens = [tokenizer.decode(t) for t in token_ids]
    
    # Header with tokenizer info
    print("\n" + "=" * 80)
    print(f"🔤 TOKENIZER: {tokenizer_name}")
    print("=" * 80)
    
    # Statistics
    if show_stats:
        special_tokens = [t for t in tokens if t.startswith('[') and t.endswith(']') or t.startswith('<|') and t.endswith('|>')]
        regular_tokens = len(tokens) - len(special_tokens)
        avg_token_length = sum(len(t) for t in tokens) / len(tokens) if tokens else 0
        
        print(f"📊 Total tokens: {len(tokens)} | Regular: {regular_tokens} | Special: {len(special_tokens)} | Avg length: {avg_token_length:.2f} chars")
        print("-" * 80)
    
    # Display tokens with enhanced visualization
    print()
    for idx, (t_id, token) in enumerate(zip(token_ids, tokens)):
        # Determine if it's a special token
        is_special = (token.startswith('[') and token.endswith(']')) or \
                     (token.startswith('<|') and token.endswith('|>')) or \
                     token in ['<s>', '</s>', '<unk>']
        
        # Choose color scheme
        if is_special:
            # Special tokens get bold yellow text on magenta background for visibility
            color_code = f'\x1b[1;93;45m'
        else:
            # Regular tokens with black text on bright colored backgrounds
            color_code = f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m'
        
        # Format token display
        token_display = token.replace('\n', '↵').replace('\t', '→').replace(' ', '·')
        
        if show_ids:
            # Show token with ID in a box-like format
            token_str = f"{color_code} {token_display} \x1b[0m"
            id_str = f"\x1b[96m[{t_id:>5}]\x1b[0m"  # Bright cyan for IDs
            print(f"{token_str}{id_str}", end=' ')
        else:
            # Show token only
            print(f"{color_code} {token_display} \x1b[0m", end=' ')
        
        # Add line break every 6 tokens for better readability
        if (idx + 1) % 6 == 0:
            print()
    
    print("\n")
    
    # Footer
    if show_stats:
        print("-" * 80)
        print("💡 Legend: Regular tokens in colors | \x1b[1;93;45m Special tokens \x1b[0m | · = space | ↵ = newline | → = tab")
        print("=" * 80 + "\n")


In [10]:
text = """
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600
"""



Link to the use Models:
- bert-base-uncased: https://huggingface.co/google-bert/bert-base-uncased
  - Tokenization method is WordPieace
  - Vocab Size = 30,522
  - Special Tokens: 
     - unk_token [UNK]: An unknown token that the tokenizer has no specific encoding for it
     - sep_token [SEP]: A separator that enables certain tasks that require giving the model two texts (in these cases, the model is called a cross-encoder).
     - pad_token [PAD]: A padding token used to pad unused positions in the model’s input (as the model expects a certain length of input, its context-size).
     - cls_token [CLS]: A special classification token for classification tasks.
     - mask_token [MASK]: A masking token used to hide tokens during the training process.


BERT was released in two major flavors: cased (where the capitalization is kept) and uncased (where all capital letters are first turned into small cap letters). With the uncased (and more popular) version of the BERT tokenizer, we notice the following:
- The newline breaks are gone, which makes the model blind to information encoded in newlines (e.g., a chat log when each turn is in a new line).All the text is in lowercase.
- The word “capitalization” is encoded as two subtokens: capital ##ization. The ## characters are used to indicate this token is a partial token connected to the token that precedes it. This is also a method to indicate where the spaces are, as it is assumed tokens without ## in front have a space before them.
- The emoji and Chinese characters are gone and replaced with the [UNK] special token indicating an “unknown token.”

In [11]:
show_tokens(text, "bert-base-uncased", show_ids=False, show_stats=True)


🔤 TOKENIZER: bert-base-uncased
📊 Total tokens: 41 | Regular: 37 | Special: 4 | Avg length: 2.93 chars
--------------------------------------------------------------------------------

 [CLS]   english   and   capital   ##ization   [UNK]  
 [UNK]   show   _   token   ##s   false  
 none   eli   ##f   =   =   >  
 =   else   :   two   tab   ##s  
 :   "   "   three   tab   ##s  
 :   "   "   12   .   0  
 *   50   =   600   [SEP]  

--------------------------------------------------------------------------------
💡 Legend: Regular tokens in colors |  Special tokens  | · = space | ↵ = newline | → = tab



### Other Tokenizer: BERT base model (cased)
- BERT base model (cased) (2018): https://huggingface.co/google-bert/bert-base-cased
- Tokenization method: WordPiece
- Vocabulary size: 28,996
- Special tokens: Same as the uncased version

In [12]:
show_tokens(text, "bert-base-cased", show_ids=False, show_stats=False)


🔤 TOKENIZER: bert-base-cased

 [CLS]   English   and   CA   ##PI   ##TA  
 ##L   ##I   ##Z   ##AT   ##ION   [UNK]  
 [UNK]   show   _   token   ##s   F  
 ##als   ##e   None   el   ##if   =  
 =   >   =   else   :   two  
 ta   ##bs   :   "   "   Three  
 ta   ##bs   :   "   "   12  
 .   0   *   50   =   600  
 [SEP]  



- Notice how “CAPITALIZATION” is now represented as eight tokens: CA ##PI ##TA ##L ##I ##Z ##AT ##ION.
- Both BERT tokenizers wrap the input within a starting [CLS] token and a closing [SEP] token. [CLS] and [SEP] are utility tokens used to wrap the input text and they serve their own purposes. [CLS] stands for classification as it’s a token used at times for sentence classification. [SEP] stands for separator, as it’s used to separate sentences in some applications that require passing two sentences to a model (For example, in Chapter 8, we will use a [SEP] token to separate the text of the query and a candidate result.)


### Other Tokenizer: GPT2
- Link to the model on the HuggingFace model hub: https://huggingface.co/openai-community/gpt2
- Tokenization method: Byte pair encoding (BPE), introduced in “Neural machine translation of rare words with subword units”.
- Vocabulary size: 50,257
- Special tokens: <|endoftext|>

In [13]:
show_tokens(text, "gpt2", show_ids=True, show_stats=True)


🔤 TOKENIZER: gpt2
📊 Total tokens: 55 | Regular: 55 | Special: 0 | Avg length: 2.29 chars
--------------------------------------------------------------------------------

 ↵ [  198]  English [15823]  ·and [  290]  ·CAP [20176]  ITAL [40579]  IZ [14887] 
 ATION [ 6234]  ↵ [  198]  � [ 8582]  � [  236]  � [  113]  ·� [16268] 
 � [  116]  � [  253]  ↵ [  198]  show [12860]  _ [   62]  t [   83] 
 ok [  482]  ens [  641]  ·False [10352]  ·None [ 6045]  ·el [ 1288]  if [  361] 
 ·== [ 6624]  ·>= [18189]  ·else [ 2073]  : [   25]  ·two [  734]  ·tabs [22524] 
 :" [11097]  · [  220]  · [  220]  · [  220]  ·" [  366]  ·Three [ 7683] 
 ·tabs [22524]  : [   25]  ·" [  366]  · [  220]  · [  220]  · [  220] 
 · [  220]  · [  220]  · [  220]  ·" [  366]  ↵ [  198]  12 [ 1065] 
 . [   13]  0 [   15]  * [    9]  50 [ 1120]  = [   28]  600 [ 8054] 
 ↵ [  198] 

--------------------------------------------------------------------------------
💡 Legend: Regular tokens in colors |  Special tokens  | · = 

With the GPT-2 tokenizer, we notice the following:
- The newline breaks are represented in the tokenizer.
- Capitalization is preserved, and the word “CAPITALIZATION” is represented in four tokens.
- The 🎵鸟 characters are now represented by multiple tokens each. While we see these tokens printed as the � character, they actually stand for different tokens. For example, the 🎵 emoji is broken down into the tokens with token IDs 8582, 236, and 113. The tokenizer is successful in reconstructing the original character from these tokens. We can see that by printing : tokenizer.decode([8582, 236, 113]), which prints out 🎵.
- The two tabs are represented as two tokens (token number 197 in that vocabulary) and the four spaces are represented as three tokens (number 220) with the final space being a part of the token for the closing quote character.
- The two tabs are represented as two tokens (token number 197 in that vocabulary) and the four spaces are represented as three tokens (number 220) with the final space being a part of the token for the closing quote character.

### Other Tokenizer: Flan-T5 (2022)
- Tokenizer method: SentencePiece: https://arxiv.org/pdf/1808.06226
- It supports BPE and the unigram language model: https://arxiv.org/abs/1804.10959
- Vocabulary size: 32,100
- Special tokens:
  - unk_token <unk>
  - pad_token <pad>



In [14]:
show_tokens(text, "google/flan-t5-small", show_ids=False, show_stats=False)


🔤 TOKENIZER: google/flan-t5-small

 English   and   CA   PI   TAL   IZ  
 ATION      <unk>      <unk>   show  
 _   to   ken   s   Fal   s  
 e   None      e   l   if  
 =   =   >   =   else   :  
 two   tab   s   :   "   "  
 Three   tab   s   :   "   "  
 12.   0   *   50   =   600  
    </s>  



The Flan-T5 family of models use  the SentencePiece method. We notice the following:
- No newline or whitespace tokens; this would make it challenging for the model to work with code.
- The emoji and Chinese characters are both replaced by the <unk> token, making the model completely blind to them.

### Other Tokenizer: GPT4 (2024)
- Tokenization method: BPE
- Vocabulary size: A little over 100,000
- Special tokens:
  - <|endoftext|>
  - Fill in the middle tokens. These three tokens enable the LLM to generate a completion given not only the text before it but also considering the text after it. This method is explained in more detail in the paper “Efficient training of language models to fill in the middle”; its exact details are beyond the scope of this book. These special tokens are:
     - <|fim_prefix|>
     - <|fim_middle|>
     - <|fim_suffix|>

In [15]:
# The official is `tiktoken` but this the same tokenizer on the HF platform
show_tokens(text, "Xenova/gpt-4", show_ids=False, show_stats=False)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.



🔤 TOKENIZER: Xenova/gpt-4

 ↵   English   ·and   ·CAPITAL   IZATION   ↵  
 �   �   �   ·�   �   �  
 ↵   show   _tokens   ·False   ·None   ·elif  
 ·==   ·>=   ·else   :   ·two   ·tabs  
 :"   ···   ·"   ·Three   ·tabs   :  
 ·"   ······   ·"↵   12   .   0  
 *   50   =   600   ↵  



# Other tokenizer startCode (2024)
- It is 15-billion parameter model focued on generating code described in the paper: https://arxiv.org/abs/2402.19173
- the original startCoder: https://arxiv.org/abs/2305.06161
- Tokenization method: Byte pair encoding (BPE)
- Vocabulary size: 49,152
-Example special tokens:
  -<|endoftext|>
  - Fill in the middle tokens: 
    - <fim_prefix>
    - <fim_middle>
    - <fim_suffix>
    - <fim_pad>
- When representing code, managing the context is important. One file might make a function call to a function that is defined in a different file.
- So the model needs some way of being able to identify code that is in different files in the same code repository, while making a distinction between code in different repos.
- That’s why StarCoder2 uses special tokens for the name of the repository and the filename:
   - <filename>
   - <reponame>
   - <gh_stars>
- 




In [17]:
show_tokens(text, "bigcode/starcoder2-15b", show_ids=False, show_stats=False)

d:\2026-courses\LLMs-Handson\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pc\.cache\huggingface\hub\models--bigcode--starcoder2-15b. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)



🔤 TOKENIZER: bigcode/starcoder2-15b

 ↵   English   ·and   ·CAPITAL   IZATION   ↵  
 �   �   �   ·   �   �  
 ↵   show   _   tokens   ·False   ·None  
 ·elif   ·==   ·>=   ·else   :   ·two  
 ·tabs   :"   ···   ·"   ·Three   ·tabs  
 :   ·"   ······   ·"   ↵   1  
 2   .   0   *   5   0  
 =   6   0   0   ↵  



- Similar to GPT4, it encodes the list if whitespaces as single token
- A major difference here to everything we’ve seen so far is that each digit is assigned its own token (so 600 becomes 6 0 0).
- The hypothesis here is that this would lead to better representation of numbers and mathematics. 
- In GPT-2, for example, the number 870 is represented as a single token. 
- But 871 is represented as two tokens (8 and 71). You can intuitively see how that might be confusing to the model and how it represents numbers.
  


# Other tokenizer: Galactica
- The Galactica model described in “Galactica: A large language model for science” is focused on scientific knowledge and is trained on many scientific papers, reference materials, and knowledge bases: https://arxiv.org/abs/2211.09085 .
- It pays extra attention to tokenization that makes it more sensitive to the nuances of the dataset it’s representing.
- For example, it includes special tokens for citations, reasoning, mathematics, amino acid sequences, and DNA sequences.
- Tokenization method: Byte pair encoding (BPE)
- Vocabulary size: 50,000
- Special tokens:
    - s (<>)
    - pad (<>)
    - s (</>)
    - <unk>
    - References: Citations are wrapped within the two special tokens: 
        - [START_REF]
        - [END_REF]
        - One example of usage from the paper is: Recurrent neural networks, long short-term memory [START_REF]Long Short-Term Memory, Hochreiter[END_REF]
    - Step-by-step reasoning:
        - <work> is an interesting token that the model uses for chain-of-thought reasoning.




In [18]:
show_tokens(text, "facebook/galactica-1.3b", show_ids=False, show_stats=False)

d:\2026-courses\LLMs-Handson\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pc\.cache\huggingface\hub\models--facebook--galactica-1.3b. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)



🔤 TOKENIZER: facebook/galactica-1.3b

 ↵   English   ·and   ·CAP   ITAL   IZATION  
 ↵   �   �   �   �   ·�  
 �   �   ↵   show   _   tokens  
 ·False   ·None   ·elif   ·   ==   ·  
 >   =   ·else   :   ·two   ·t  
 abs   :   "   ····   "   ·Three  
 ·t   abs   :   ·   "   ·······  
 "   ↵   1   2   .   0  
 *   5   0   =   6   0  
 0   ↵  



- The Galactica tokenizer behaves similar to StarCoder2 in that it has code in mind. 
- It also encodes whitespaces in the same way: assigning a single token to sequences of whitespace of different lengths. 
- It differs in that it also does that for tabs, though. 
- So from all the tokenizers we’ve seen so far, it’s the only one that assigns a single token to the string made up of two tabs ('\t\t').

# Other tokenizer: Phi-3 (and Llama 2)
- The Phi-3 model we look at in this book reuses the tokenizer of Llama 2 yet adds a number of special tokens.
- Tokenization method: Byte pair encoding (BPE)
- Vocabulary size: 32,000
- Special tokens:
    - <|endoftext|>
    - Chat tokens: As chat LLMs rose to popularity in 2023, the conversational nature of LLMs started to be a leading use case. Tokenizers have been adapted to this direction by the addition of tokens that indicate the turns in a conversation and the roles of each speaker. These special tokens include:
         - <|user|>
         - |assistant|>
         - <|system|>


In [20]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct", show_ids=False, show_stats=False)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.



🔤 TOKENIZER: microsoft/Phi-3-mini-4k-instruct

    ↵   English   and   C   AP  
 IT   AL   IZ   ATION   ↵   �  
 �   �   �      �   �  
 �   ↵   show   _   to   kens  
 False   None   elif   ==   >=   else  
 :   two   tabs   :"   ··   "  
 Three   tabs   :   "   ·····   "  
 ↵   1   2   .   0   *  
 5   0   =   6   0   0  
 ↵  

